<a href="https://colab.research.google.com/github/FatherNurt/FPDP-Engine/blob/MAIN/C9_Solar_Weather_Input_Provenance_Audit_v0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C9 Solar Weather Input-Provenance Audit v0.1
Standalone reproducibility notebook. **Does not modify FPDP or C9.** Processed CSV coverage audit only; raw feed provenance requires a separate adapter instrumented run. ZIP is saved in Files sidebar without triggering an iPad browser download.

In [ ]:
import pandas as pd, json, hashlib, pathlib, zipfile
from IPython.display import display
p=pathlib.Path('fpdp_history.csv')
if not p.exists():
    try:
        from google.colab import files
        up=files.upload()
        p=pathlib.Path(next(iter(up)))
    except ImportError:
        raise FileNotFoundError('Place fpdp_history.csv next to notebook')
d=pd.read_csv(p)
fields=['magnetic_perturbation','electron_disturbance','bz_southward_stress','solar_wind_speed_stress','solar_wind_density_stress','l1_lag_minutes_used','observed_kp_validation_only']
rows=[{'feature':f,'column_present':f in d.columns,'non_null_rows':int(d[f].notna().sum()) if f in d else 0,'coverage_pct':round(100*d[f].notna().mean(),2) if f in d else 0.0} for f in fields]
t=pd.DataFrame(rows); display(t)
complete=all((f in d.columns and d[f].notna().all()) for f in fields[:5]) and 'l1_lag_minutes_used' in d and d.l1_lag_minutes_used.notna().all()
status='COVERAGE_COMPLETE_PROVENANCE_UNVERIFIED' if complete else 'PARTIAL_COVERAGE_NOT_ELIGIBLE_FOR_FULL_SOLAR_WIND_VALIDATION'
manifest={'status':status,'input_sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'rows':len(d),'coverage':rows,'caution':'Processed history cannot prove native feed timestamps, raw missingness, or correct lag alignment.'}
out=pathlib.Path('C9_Provenance_Audit_v0_1');out.mkdir(exist_ok=True)
t.to_csv(out/'coverage.csv',index=False)
(out/'manifest.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile('C9_Provenance_Audit_v0_1.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir():z.write(f,f.name)
print('Saved C9_Provenance_Audit_v0_1.zip in Colab Files sidebar. Status:',status)
